# Notebook 1 — Preparación de datos: DisTEMIST

Objetivo: cargar el corpus DisTEMIST (NER + normalización de enfermedades a SNOMED-CT, en español)
y dividirlo en splits train/dev/test para que la comparación entre modelos en el fine tuning sea justa.


## 1. Setup e imports

Cargar datos crudos desde carpeta en drive.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")

except ImportError:
    RAW_DATA_DIR = "."

Mounted at /content/drive


In [ ]:
# !pip install datasets huggingface_hub --quiet

import json
import os
import random
import numpy as np
from pathlib import Path
from collections import Counter

from datasets import Dataset, DatasetDict

random.seed(42)

# ─────────────────────────────────────────────────────────────────────────
# RUTA BASE EN GOOGLE DRIVE (Mi unidad > TopicosIA)
BASE_DIR = Path("/content/drive/MyDrive/TopicosIA")
# ─────────────────────────────────────────────────────────────────────────

RAW_DATA_DIR = BASE_DIR / "distemist_raw"
FINAL_DATA_DIR = BASE_DIR / "distemist_final"
FINAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Si es True, usa un dataset sintético pequeño para poder correr el notebook
# de punta a punta sin conexión (útil para probar el pipeline antes de bajar
# el corpus real).
USE_MOCK_DATA = False

## 2. Carga de DisTEMIST

Se carga el dataset en su formato original (desde la página Zenodo), este incluye texto plano y archivos tsv.


In [ ]:
import re

def fix_unicode_escapes(text):
    return re.sub(r'\\u([0-9a-fA-F]{4})', lambda m: chr(int(m.group(1), 16)), text)

In [ ]:
# Desde Zenodo, formato real (texto plano + TSV)
# TSV subtrack2_linking, columnas: filename, mark, label, off0, off1, span, codes, semantic_relation
# 'codes' puede traer varios códigos SNOMED-CT concatenados con "+"
def load_from_tsv(raw_dir=RAW_DATA_DIR, txt_subdir="text_files",
                   tsv_dir="subtrack1_entities", max_docs=None, seed=42):
    import csv
    import glob
    import random

    raw_dir = Path(raw_dir)
    # Soporte flexible si los datos están directamente en distemist_raw o dentro de training/
    if (raw_dir / "training").exists():
        raw_dir = raw_dir / "training"

    txt_dir = raw_dir / txt_subdir if (raw_dir / txt_subdir).exists() else raw_dir
    tsv_paths = sorted(glob.glob(str(raw_dir / tsv_dir / "*.tsv")))
    if not tsv_paths:
        tsv_paths = sorted(glob.glob(str(raw_dir / "*.tsv")))

    if not tsv_paths:
        raise FileNotFoundError(f"No se encontraron .tsv en {raw_dir / tsv_dir} ni en {raw_dir}")

    # Agrupar anotaciones por documento, leyendo todos los .tsv de la carpeta
    anns_by_doc = {}
    for tsv_path in tsv_paths:
        with open(tsv_path, encoding="utf-8") as f:
            reader = csv.DictReader(f, delimiter="\t")
            for row in reader:
                doc_id = row["filename"]
                anns_by_doc.setdefault(doc_id, []).append({
                    "start": int(row["off0"]),
                    "end": int(row["off1"]),
                    "text": fix_unicode_escapes(row["span"]),
                    "label": row.get("label", "ENFERMEDAD"),
                })

    txt_paths = sorted(txt_dir.glob("*.txt"))
    if not txt_paths:
        raise FileNotFoundError(f"No se encontraron .txt en {txt_dir}")

    # Controlar cuántos documentos se cargan, con muestreo reproducible
    if max_docs is not None and max_docs < len(txt_paths):
        txt_paths = random.Random(seed).sample(txt_paths, max_docs)
        txt_paths = sorted(txt_paths)

    records = []
    for txt_path in txt_paths:
        doc_id = txt_path.stem
        text = txt_path.read_text(encoding="utf-8")
        entities = anns_by_doc.get(doc_id, [])
        records.append({"doc_id": doc_id, "text": text, "entities": entities})
    return records

In [ ]:
# --- Datos sintéticos de respaldo (para probar el notebook sin internet) ---
MOCK_RECORDS = [
    {
        "doc_id": "mock_001",
        "text": "Paciente varón de 68 años que ingresa por disnea progresiva y tos con expectoracion. "
                "En la exploracion se objetiva neumonia adquirida en la comunidad. Antecedente de diabetes mellitus tipo 2.",
        "entities": [
            {"start": 108, "end": 143, "text": "neumonia adquirida en la comunidad",
             "label": "ENFERMEDAD", "normalized_term": "Neumonía adquirida en la comunidad"},
            {"start": 166, "end": 189, "text": "diabetes mellitus tipo 2",
             "label": "ENFERMEDAD", "normalized_term": "Diabetes mellitus tipo 2"},
        ],
    },
    {
        "doc_id": "mock_002",
        "text": "Mujer de 45 años con antecedentes de hipertension arterial que consulta por cefalea intensa "
                "de una semana de evolucion, sin fiebre asociada.",
        "entities": [
            {"start": 38, "end": 59, "text": "hipertension arterial",
             "label": "ENFERMEDAD", "normalized_term": "Hipertensión arterial"},
        ],
    },
]

En la celda de abajo cambiar `max_docs` por el número de documentos a usar, en caso de que se quiera cargar todo el dataset, no pasar ningún argumento a la función load_from_tsv().

In [ ]:
def load_mock_data():
    return MOCK_RECORDS

if USE_MOCK_DATA:
    raw_records = load_mock_data()
else:
    raw_records = load_from_tsv(max_docs=30)

print(f"Documentos cargados: {len(raw_records)}")

Documentos cargados: 30


In [ ]:
def check_docs_without_entities(records):
    docs_without_entities = [rec["doc_id"] for rec in records if len(rec["entities"]) == 0]

    print(f"Total de documentos: {len(records)}")
    print(f"Documentos sin entidades: {len(docs_without_entities)} "
          f"({100 * len(docs_without_entities) / len(records):.1f}%)")

    if docs_without_entities:
        print("\nEjemplos de doc_id afectados:")
        for doc_id in docs_without_entities[:10]:
            print(" -", doc_id)

    return docs_without_entities

missing = check_docs_without_entities(raw_records)

Total de documentos: 30
Documentos sin entidades: 0 (0.0%)


## 3. Prueba de tokenizadores

En esta serie de pruebas se utilizan 4 tokenizadores como un punto de partida para poder entender su funcionamiento ("qué tenemos antes de tocar nada"). En futuros pasos puede que se conserven o no como tokenizadores oficiales para el pipeline de fine tuning.

Set up de pruebas:

In [ ]:
# Diagnóstico: qué tenemos ANTES de tocar nada
import importlib.metadata as md_, sys

print('python', sys.version.split()[0])
for p in ['torch', 'transformers', 'tokenizers', 'sentencepiece', 'pandas']:
    try:
        print(f'{p:16} {md_.version(p)}')
    except md_.PackageNotFoundError:
        print(f'{p:16} FALTA')

python 3.12.13
torch            2.11.0+cu128
transformers     5.13.1
tokenizers       0.22.2
sentencepiece    0.2.2
pandas           2.2.2


In [ ]:
try:
    import transformers, sentencepiece  # noqa: F401
    print('Entorno completo — no instalo nada.')
except ImportError:
    %pip -q install transformers sentencepiece
    print('Instalado. Si Colab pide RESTART RUNTIME, reinicia y vuelve a correr desde aquí.')

Entorno completo — no instalo nada.


In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import AutoTokenizer

torch.manual_seed(42)
pd.set_option('display.max_colwidth', None)
print('torch', torch.__version__, '| listo')

torch 2.11.0+cu128 | listo


In [ ]:
TOKENIZADORES = {
    'BERT-en':'bert-base-uncased',
    'BERT-clinico-en': 'medicalai/ClinicalBERT',
    'Roberta-clinico-es': 'BSC-TeMU/roberta-base-biomedical-es',
    'mT5': 'google/mt5-small',
}

toks = {}
for nombre, ruta in TOKENIZADORES.items():
    toks[nombre] = AutoTokenizer.from_pretrained(ruta)
    print(f'{nombre:24} vocabulario: {toks[nombre].vocab_size:>7,} tokens')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BERT-en                  vocabulario:  30,522 tokens


...

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

mT5                      vocabulario: 250,100 tokens


Visualizemos cómo cada tokenizador despedaza el mismo texto.

In [ ]:
def comparar(texto):
    """Muestra cómo cada tokenizador despedaza el mismo texto."""
    filas = []
    for nombre, tk in toks.items():
        piezas = tk.tokenize(texto)
        filas.append({'Tokenizador': nombre, 'N': len(piezas), 'Tokens': ' | '.join(piezas)})
    return pd.DataFrame(filas).set_index('Tokenizador')

comparar('Varón de 35 años, con antecedentes de lúes, hepatitis no filiada y circuncisión a los 4 años, que consulta por bifidez del chorro miccional.')

,N,Tokens
Tokenizador,,
BERT-en,46,"var | ##on | de | 35 | an | ##os | , | con | ant | ##ece | ##dent | ##es | de | lu | ##es | , | hepatitis | no | fi | ##lia | ##da | y | ci | ##rc | ##un | ##cision | a | los | 4 | an | ##os | , | que | consult | ##a | por | bi | ##fide | ##z | del | cho | ##rro | mic | ##cion | ##al | ."
BERT-clinico-en,42,"var | ##on | de | 35 | anos | , | con | ante | ##cedent | ##es | de | lu | ##es | , | hep | ##ati | ##tis | no | fil | ##iada | y | ci | ##rc | ##un | ##cision | a | los | 4 | anos | , | que | consulta | por | bi | ##fi | ##dez | del | cho | ##rro | mic | ##cional | ."
Roberta-clinico-es,32,"ĠVarÃ³n | Ġde | Ġ35 | ĠaÃ±os | , | Ġcon | Ġantecedentes | Ġde | ĠlÃº | es | , | Ġhepatitis | Ġno | Ġfil | iada | Ġy | ĠcircuncisiÃ³n | Ġa | Ġlos | Ġ4 | ĠaÃ±os | , | Ġque | Ġconsulta | Ġpor | Ġbi | fi | dez | Ġdel | Ġchorro | Ġmiccional | ."
mT5,43,"▁Var | ón | ▁de | ▁35 | ▁años | , | ▁con | ▁ante | cedente | s | ▁de | ▁lú | es | , | ▁ | hepatit | is | ▁no | ▁fili | ada | ▁ | y | ▁circun | ci | sión | ▁ | a | ▁los | ▁4 | ▁años | , | ▁que | ▁consulta | ▁por | ▁bi | f | idez | ▁del | ▁c | horro | ▁mic | cional | ."


Se puede observar que Roberta tiene una ventaja considerable frente a los demás tokenizadores.

A continuación calculamos estadísticas de longitud de secuencias  tokens para cada tokenizador.

In [ ]:
def measure_tokens(records, tokenizer):
    """Calcular longitud de cadena de tokens para textos clínicos."""
    lengths = []
    for rec in records:
        lengths.append(len(tokenizer.encode(rec["text"], add_special_tokens=True)))
    return lengths


def calculate_lengths(toks, record):
    """Calcular estadísticas de longitud de tokens."""
    for tokenizer_name, tokenizer in toks.items():
        lengths = measure_tokens(record, tokenizer)
        print(f"{tokenizer_name}:")
        print("Máximo:", max(lengths))
        print("Media:", np.mean(lengths))
        print("Mediana:", np.median(lengths))
        print("P70:", np.percentile(lengths, 70))
        print("P99:", np.percentile(lengths, 99))
        print('\n')

In [ ]:
calculate_lengths(toks, raw_records)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1317 > 512). Running this sequence through the model will result in indexing errors


BERT-en:
Máximo: 2125
Media: 932.1333333333333
Mediana: 808.5
P70: 1015.9
P99: 2073.67




[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (750 > 512). Running this sequence through the model will result in indexing errors


BERT-clinico-en:
Máximo: 1793
Media: 767.0666666666667
Mediana: 671.0
P70: 840.5999999999999
P99: 1748.0500000000002


Roberta-clinico-es:
Máximo: 1293
Media: 560.9
Mediana: 507.5
P70: 593.4
P99: 1286.91


mT5:
Máximo: 1841
Media: 805.1333333333333
Mediana: 713.0
P70: 865.8
P99: 1783.8700000000001




Como se puede observar en la anterior salida por consola, la cantidad de tokens por secuencia de texto varía mucho entre tokenizadores. Esto es un obstáculo considerable a la hora de preparar los datos, ya que para todos los tokenizadores el 70% de los textos se convierten en secuencias de tokens de longitud mayor a 593.

Teniendo en cuenta que la arquitectura de los modelos propuestos soporta hasta 512 tokens por secuencia, decidimos implementar chunking con solapamiento (para no perder las entidades en los textos y tener el tamaño adecuado) en la siguiente etapa. Por ello, en este notebook se hace la separación en splits (train/dev/test) que será el punto de entrada de la etapa de chunking y formateo de datos para fine tuning de cada arquitectura.

Extra: Explorar el ratio de tokens/palabras para cada tokenizador.

In [ ]:
def compute_safe_window_words(tokenizer_name, tokenizer, texts, target_max_tokens=480, safety_margin=0.9):
    """
    target_max_tokens: tope real que quieres respetar (dejamos margen bajo 512
                        para [CLS]/[SEP] y para no quedar justo en el límite).
    safety_margin: reduce el resultado un poco más, para cubrir documentos
                   con vocabulario más fragmentado que el promedio de la muestra.
    """
    ratios = []
    for text in texts:
        n_words = len(__import__("re").findall(r"\S+", text))
        n_tokens = len(tokenizer.encode(text, add_special_tokens=False))
        if n_words > 0:
            ratios.append(n_tokens / n_words)

    # usamos el percentil 90 de fragmentación, no el promedio, para no
    # subestimar documentos con vocabulario más raro/fragmentado que el típico
    worst_case_ratio = np.percentile(ratios, 90)
    safe_window = int((target_max_tokens / worst_case_ratio) * safety_margin)

    print("\nTokenizador: ", tokenizer_name)
    print(f"Ratio tokens/palabra (p90): {worst_case_ratio:.2f}")
    print(f"window_words seguro para este tokenizer: {safe_window}")
    return safe_window

In [ ]:
sample_texts = [rec["text"] for rec in raw_records]  # documentos originales, sin chunkear

windows = {}

for tokenizer_name, tokenizer in toks.items():
  windows[tokenizer_name] = compute_safe_window_words(tokenizer_name, tokenizer, sample_texts)


Tokenizador:  BERT-en
Ratio tokens/palabra (p90): 2.35
window_words seguro para este tokenizer: 183

Tokenizador:  BERT-clinico-en
Ratio tokens/palabra (p90): 1.97
window_words seguro para este tokenizer: 218

Tokenizador:  Roberta-clinico-es
Ratio tokens/palabra (p90): 1.42
window_words seguro para este tokenizer: 304

Tokenizador:  mT5
Ratio tokens/palabra (p90): 2.05
window_words seguro para este tokenizer: 210


Esto confirma que se debe dar un tratamiento diferente a los datos de entrada para cada modelo.

## 4. Split train / dev / test


In [ ]:
def make_split(records, train_ratio=0.8, dev_ratio=0.1, seed=42):
    records = records[:]
    random.Random(seed).shuffle(records)
    n = len(records)
    n_train = max(1, int(n * train_ratio))
    n_dev = max(1, int(n * dev_ratio))
    return {
        "train": records[:n_train],
        "dev": records[n_train:n_train + n_dev],
        "test": records[n_train + n_dev:] or records[-1:],  # asegura al menos 1 ejemplo
    }

splits = make_split(raw_records)
for name, subset in splits.items():
    print(name, len(subset))

train 24
dev 3
test 3


In [ ]:
# Visualizar contenido de cada split
for split_name, records in splits.items():
    print(f"{split_name}:")
    print(json.dumps(records, indent=2, ensure_ascii=False))

train:
[
  {
    "doc_id": "es-S1130-05582015000100005-1",
    "text": "Un varón de 30 años de edad sufrió un accidente durante la carga de un fusil de pesca submarina en su casa. Se avisó a emergencias. Después del protocolo \"A, B, C\" el examen clínico del paciente mostraba únicamente una herida penetrante a nivel de la región submandibular.\nEl tamaño aproximado del arpón era de 80 cm de largo y 1 cm de diámetro con un trayecto intracraneal de unos 15 cm. Los bomberos cortaron cuidadosamente el arpón para facilitar el traslado del paciente al hospital.\nEl paciente estaba hemodinámicamente estable, alerta y orientado (Glasgow 15), sin déficit neurológico y sin sangrado activo. Presentaba coágulos de sangre en el canal auditivo izquierdo.\nNo había salida de líquido cefalorraquídeo por la herida de entrada ni por el conducto auditivo externo. La radiografía lateral de cráneo mostraba la trayectoria, la dirección y la existencia de mecanismo de barba en el arpón. Se realizó un TAC de

### 5. Guardar splits crudos

Se guardan para que el Notebook 2 los cargue directamente y continuar con chunking y formateo para fine tuning.

Se almacenan en archivos .arrow (Apache Arrow). Este formato permite guardar y transferir grandes volúmenes de información de manera muy rápida sin necesidad de hacer conversiones pesadas.


In [ ]:
raw_splits = DatasetDict({
    split_name: Dataset.from_list(records)  # doc_id, text, entities (con offsets originales)
    for split_name, records in splits.items()
})
raw_splits.save_to_disk(str(FINAL_DATA_DIR / "distemist_raw_splits"))
print(f"Splits guardados exitosamente en: {FINAL_DATA_DIR / 'distemist_raw_splits'}")

Saving the dataset (0/1 shards):   0%|          | 0/24 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3 [00:00<?, ? examples/s]

Splits guardados exitosamente en: /content/drive/MyDrive/TopicosIA/distemist_final/distemist_raw_splits
